<div dir="rtl">

# نوت‌بوک ۵: تحلیل اکتشافی

**فاز ۱، تسک‌های ۸ و ۹**

سند دو نوع تحلیل اکتشافی می‌خواهد:

- **تک‌متغیره (تسک ۸):** برای متغیرهای اصلی، توزیع، مرکز، پراکندگی، عدم‌توازن و شکل توزیع با آمار خلاصه و نمودار مناسب.
- **چندمتغیره (تسک ۹):** چند رابطه‌ی مهم میان متغیرها، با نمودارهایی که الگو، تعامل یا تفاوت زیرگروه‌ها را نشان دهند.

**متغیر هدف ما امتیاز است.** در بخش چندمتغیره، معیار اصلی را **«سهم نقدهای منفی»** (۱ و ۲ ستاره) می‌گیریم، چون گزارش ما درباره‌ی شکایت‌هاست. میانگین امتیاز را هم کنارش می‌آوریم.

همه‌ی تحلیل‌ها پس از حذف نقدهای کاملاً تکراری (تصمیم نوت‌بوک ۲) انجام می‌شوند. هیچ‌کدام ادعای علّی نیست؛ داده مشاهده‌ای است.

</div>

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from text_utils import clean_text, word_count

cfg = yaml.safe_load(open(ROOT / "configs" / "config.yaml", encoding="utf-8"))
PROC = ROOT / cfg["data"]["processed_dir"]
FIG = ROOT / "figures"

pd.set_option("display.width", 200)
BAR, NEG = "#3B6EA8", "#B5523B"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "font.size": 9,
})

reviews = pd.read_parquet(PROC / "reviews_rc.parquet").drop_duplicates().reset_index(drop=True)
meta = pd.read_parquet(PROC / "meta_rc.parquet")

# product-level helper columns
meta["subgroup"] = meta["categories"].map(lambda c: (json.loads(c) + [None] * 3)[2] or "(none)")
meta["log10_price"] = np.log10(meta["price"])
n_rev = reviews.groupby("parent_asin").size()
meta["n_reviews"] = meta["parent_asin"].map(n_rev).fillna(0).astype(int)

# review-level helper columns
reviews["date"] = pd.to_datetime(reviews["timestamp"], unit="ms")
reviews["year"] = reviews["date"].dt.year
reviews["words"] = reviews["text"].map(lambda t: word_count(clean_text(t)))
reviews["negative"] = reviews["rating"] <= 2
reviews = reviews.merge(meta[["parent_asin", "subgroup", "price", "n_reviews"]], on="parent_asin", how="left")
print(f"reviews: {len(reviews):,}   products: {len(meta):,}")

<div dir="rtl">

# بخش اول: تحلیل تک‌متغیره

## ۱. امتیاز (متغیر هدف)

</div>

In [ ]:
r = reviews["rating"].value_counts().sort_index()
rating_table = pd.DataFrame({"reviews": r, "share_%": (100 * r / r.sum()).round(1)})
print(rating_table.to_string())
print()
print(f"mean = {reviews['rating'].mean():.2f}   median = {reviews['rating'].median():.0f}   "
      f"std = {reviews['rating'].std():.2f}")
print(f"imbalance: 5-star reviews are {r[5] / r[2]:.1f}x as many as 2-star reviews")
print(f"share negative (1-2 stars): {reviews['negative'].mean():.1%}")
print(f"a model that always says '5 stars' would be right {r[5] / r.sum():.1%} of the time")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
colors = [NEG, NEG, "#9a9a9a", BAR, BAR]
ax.bar(rating_table.index.astype(int).astype(str), rating_table["share_%"], color=colors, width=0.6)
for x, v in zip(range(5), rating_table["share_%"]):
    ax.text(x, v + 1, f"{v:.1f}%", ha="center", fontsize=8.5)
ax.set_xlabel("stars")
ax.set_ylabel("% of reviews")
ax.set_ylim(0, 75)
ax.set_title("Star ratings: J-shaped and dominated by 5 stars", loc="left")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.savefig(FIG / "fig08_rating_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۲. خلاصه‌ی آماری متغیرهای عددی اصلی

برای هر متغیر، علاوه بر میانگین و انحراف معیار، **میانه** و **چولگی** را هم می‌آوریم. چولگی مثبت و بزرگ یعنی دنباله‌ی بلند به سمت راست؛ در این حالت میانه از میانگین گویاتر است.

</div>

In [ ]:
def describe(s, name, level):
    s = s.dropna()
    return {"variable": name, "level": level, "n": len(s),
            "mean": s.mean(), "median": s.median(), "std": s.std(),
            "p10": s.quantile(.1), "p90": s.quantile(.9), "max": s.max(),
            "skewness": s.skew()}

summary = pd.DataFrame([
    describe(reviews["rating"], "rating", "review"),
    describe(reviews["words"], "words per review", "review"),
    describe(reviews["helpful_vote"], "helpful votes", "review"),
    describe(meta["price"], "price (USD)", "product"),
    describe(meta["n_reviews"], "reviews per product", "product"),
]).set_index("variable").round(2)
summary

In [ ]:
cats = pd.DataFrame({
    "verified purchase": [f"{reviews['verified_purchase'].mean():.1%} of reviews"],
    "review has image": [f"{(reviews['n_images'] > 0).mean():.1%} of reviews"],
    "product has price": [f"{meta['price'].notna().mean():.1%} of products"],
}).T.rename(columns={0: "share"})
cats

<div dir="rtl">

## ۳. قیمت

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(meta["log10_price"].dropna(), bins=50, color=BAR)
ticks = [1, 3, 10, 30, 100, 300, 1000, 3000]
ax.set_xticks(np.log10(ticks), [f"${t:,}" for t in ticks])
ax.axvline(np.log10(meta["price"].median()), color="#333", lw=1, ls="--")
ax.text(np.log10(meta["price"].median()) + 0.04, ax.get_ylim()[1] * 0.9,
        f"median ${meta['price'].median():.0f}", fontsize=8.5)
ax.set_xlabel("price (log scale)")
ax.set_ylabel("products")
ax.set_title(f"Product price ({meta['price'].notna().sum():,} products with a price)", loc="left")
plt.tight_layout()
plt.savefig(FIG / "fig09_price_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۴. تعداد نقد در طول زمان

</div>

In [ ]:
monthly = reviews.set_index("date").resample("MS").size()
monthly = monthly[monthly.index >= "2012-01-01"]

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(monthly.index, monthly.values, color=BAR, lw=1.6)
ax.axvspan(pd.Timestamp("2023-01-01"), monthly.index.max(), color="#f3d9d3", alpha=0.6, lw=0)
ax.text(pd.Timestamp("2023-01-15"), monthly.max() * 0.92, "2023: likely\nincomplete", fontsize=8, color="#7a3424")
ax.set_ylabel("reviews per month")
ax.set_title("Reviews per month", loc="left")
plt.tight_layout()
plt.savefig(FIG / "fig10_reviews_per_month.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

در این نمودار دو الگو دیده می‌شود: **رشد بلندمدت** تا اوج ۲۰۲۰ و ۲۰۲۱، و **قله‌های تکرارشونده** که احتمالاً به فصل خرید پایان سال (کریسمس) مربوط‌اند؛ اسباب‌بازی کالای هدیه است. افت ماه‌به‌ماه ۲۰۲۳ هم که قبلاً دیدیم، اینجا واضح است.

</div>

In [ ]:
by_month = reviews[reviews["year"].between(2015, 2022)].groupby(reviews["date"].dt.month).size()
share_by_month = (100 * by_month / by_month.sum()).round(1)
print("share of 2015-2022 reviews by calendar month (%):")
print(share_by_month.to_string())

<div dir="rtl">

# بخش دوم: تحلیل چندمتغیره

در هر رابطه می‌پرسیم: **سهم نقدهای منفی** در گروه‌های مختلف چقدر فرق دارد؟

## ۵. امتیاز در زیرگروه‌های محصول

</div>

In [ ]:
by_sub = (reviews.groupby("subgroup")
                 .agg(reviews=("rating", "size"), mean_rating=("rating", "mean"),
                      negative_pct=("negative", "mean"))
                 .query("reviews >= 3000")
                 .sort_values("negative_pct"))
by_sub["negative_pct"] = (100 * by_sub["negative_pct"]).round(1)
by_sub["mean_rating"] = by_sub["mean_rating"].round(2)
by_sub

<div dir="rtl">

## ۶. امتیاز و قیمت

محصولات قیمت‌دار را به پنج گروه هم‌اندازه (پنجک) تقسیم می‌کنیم و محصولات بی‌قیمت را گروه ششم در نظر می‌گیریم.

</div>

In [ ]:
priced = meta["price"].notna()
meta["price_band"] = "no price"
meta.loc[priced, "price_band"] = pd.qcut(meta.loc[priced, "price"], 5,
                                         labels=["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 priciest"]).astype(str)
reviews["price_band"] = reviews["parent_asin"].map(meta.set_index("parent_asin")["price_band"])
band_edges = meta.loc[priced].groupby("price_band")["price"].agg(["min", "max"]).round(2)

by_price = (reviews.groupby("price_band")
                   .agg(reviews=("rating", "size"), mean_rating=("rating", "mean"),
                        negative_pct=("negative", "mean")))
by_price["negative_pct"] = (100 * by_price["negative_pct"]).round(1)
by_price["mean_rating"] = by_price["mean_rating"].round(2)
order = ["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 priciest", "no price"]
by_price = by_price.reindex(order).join(band_edges)
by_price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
ax.barh(by_sub.index, by_sub["negative_pct"], color=NEG, height=0.6)
ax.axvline(100 * reviews["negative"].mean(), color="#555", lw=1, ls="--")
ax.set_title("By product sub-group", loc="left")
ax.set_xlabel("% of reviews that are 1-2 stars")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="y", visible=False)

ax = axes[1]
ax.bar(by_price.index, by_price["negative_pct"], color=[NEG] * 5 + ["#9a9a9a"], width=0.6)
ax.axhline(100 * reviews["negative"].mean(), color="#555", lw=1, ls="--")
ax.set_title("By price band (quintiles)", loc="left")
ax.set_ylabel("% of reviews that are 1-2 stars")
ax.tick_params(axis="x", labelsize=8)
ax.grid(axis="x", visible=False)
fig.suptitle("Share of negative reviews (dashed line = overall)", x=0.01, ha="left", fontsize=10)
plt.tight_layout()
plt.savefig(FIG / "fig11_negative_by_subgroup_price.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۷. امتیاز در طول زمان

آیا مشتری‌ها در طول سال‌ها راضی‌تر یا ناراضی‌تر شده‌اند؟ این برای ما مهم است، چون اگر سطح کلی نارضایتی در طول زمان عوض شود، مدلی که روی گذشته آموزش دیده ممکن است روی آینده خطا کند.

</div>

In [ ]:
by_year = (reviews[reviews["year"] >= 2012].groupby("year")
                  .agg(reviews=("rating", "size"), mean_rating=("rating", "mean"),
                       negative_pct=("negative", "mean")))
by_year["negative_pct"] = (100 * by_year["negative_pct"]).round(1)
by_year["mean_rating"] = by_year["mean_rating"].round(2)
by_year.T

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(by_year.index, by_year["negative_pct"], color=NEG, lw=2, marker="o", ms=5)
ax.axvspan(2022.5, 2023.5, color="#f3d9d3", alpha=0.6, lw=0)
ax.text(2022.6, by_year["negative_pct"].max() * 0.55, "2023:\nincomplete", fontsize=8, color="#7a3424")
ax.set_ylim(0, by_year["negative_pct"].max() * 1.15)
ax.set_xticks(by_year.index)
ax.set_ylabel("% of reviews that are 1-2 stars")
ax.set_title("Share of negative reviews by year", loc="left")
plt.tight_layout()
plt.savefig(FIG / "fig13_negative_by_year.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۸. تعامل: آیا رابطه‌ی قیمت و نارضایتی در همه‌ی زیرگروه‌ها یکسان است؟

«تعامل» یعنی اثر یک متغیر بسته به متغیر دیگر فرق کند. اینجا برای سه زیرگروه بزرگ (وسیله‌ی کامل، قطعه، پهپاد) جداگانه رابطه‌ی قیمت و سهم نقد منفی را می‌کشیم. پنجک‌های قیمت **درون هر زیرگروه** ساخته می‌شوند، چون یک پهپاد ارزان و یک قطعه‌ی ارزان قیمت‌های خیلی متفاوتی دارند.

</div>

In [ ]:
big3 = ["Remote & App Controlled Vehicles", "Remote & App Controlled Vehicle Parts",
        "Quadcopters & Multirotors"]
short = {big3[0]: "Vehicles", big3[1]: "Vehicle parts", big3[2]: "Quadcopters"}

rows = []
for g in big3:
    m = meta[(meta["subgroup"] == g) & meta["price"].notna()].copy()
    m["q"] = pd.qcut(m["price"], 5, labels=[1, 2, 3, 4, 5])
    rv = reviews[reviews["subgroup"] == g].merge(m[["parent_asin", "q"]], on="parent_asin")
    t = rv.groupby("q", observed=True)["negative"].agg(["mean", "size"])
    for q, row in t.iterrows():
        rows.append({"subgroup": short[g], "price_quintile": int(q),
                     "negative_pct": 100 * row["mean"], "reviews": int(row["size"])})
inter = pd.DataFrame(rows)
inter.pivot(index="price_quintile", columns="subgroup", values="negative_pct").round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, (name, d) in zip(axes, inter.groupby("subgroup", sort=False)):
    ax.plot(d["price_quintile"], d["negative_pct"], color=NEG, lw=2, marker="o", ms=5)
    ax.set_title(name, loc="left")
    ax.set_xticks([1, 2, 3, 4, 5], ["cheapest", "2", "3", "4", "priciest"])
    ax.set_xlabel("price quintile within sub-group")
axes[0].set_ylabel("% of reviews that are 1-2 stars")
axes[0].set_ylim(0, None)
fig.suptitle("Price vs negative share, separately for each sub-group", x=0.01, ha="left", fontsize=10)
plt.tight_layout()
plt.savefig(FIG / "fig12_price_subgroup_interaction.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۹. سایر روابط: خرید تأییدشده، محبوبیت و طول متن

</div>

In [ ]:
def neg_table(col, labels=None):
    t = reviews.groupby(col, observed=True).agg(
        reviews=("rating", "size"), mean_rating=("rating", "mean"), negative_pct=("negative", "mean"))
    t["negative_pct"] = (100 * t["negative_pct"]).round(1)
    t["mean_rating"] = t["mean_rating"].round(2)
    return t

reviews["popularity"] = pd.cut(reviews["n_reviews"], [0, 5, 20, 100, 1000, np.inf],
                               labels=["1-5", "6-20", "21-100", "101-1000", "1000+"])
reviews["length_band"] = pd.cut(reviews["words"], [-1, 3, 9, 50, 200, np.inf],
                                labels=["0-3", "4-9", "10-50", "51-200", "200+"])
print("verified purchase:");          display(neg_table("verified_purchase"))
print("product popularity (reviews per product):"); display(neg_table("popularity"))
print("review length (words):");     display(neg_table("length_band"))

<div dir="rtl">

## ۱۰. جمع‌بندی

_این بخش را بعد از اجرای نوت‌بوک و دیدن خروجی‌ها با هم می‌نویسیم._

</div>